# Experiment: YOLOv8-P2 Baselines on Google Colab (D0-D2)

Notebook chạy trọn vẹn giai đoạn chuẩn hóa dữ liệu (`D0`), kiểm tra tải GPU (`D1`) và huấn luyện 3 baseline (`B0-Raw`, `B1-Raw`, `B0-Enhanced` — `D2`) theo đúng giao thức trong `PLAN.md`.

**Nguyên tắc quan trọng trên Colab:**
- Dữ liệu ảnh được giải nén/sao chép ra ổ SSD cục bộ `/content/` để đọc nhanh khi train.
- Toàn bộ `runs/` (checkpoint `last.pt`, `best.pt`, `training_telemetry.json`) và `reports/` được ghi trực tiếp vào **Google Drive** để không mất tiến trình khi phiên Colab ngắt kết nối.


In [ ]:
# Bước 1: Mount Google Drive và cấu hình đường dẫn
from pathlib import Path
import os

from google.colab import drive
drive.mount('/content/drive')

# Chỉnh lại 3 đường dẫn dưới đây theo vị trí bạn upload trên Google Drive:
DRIVE_ROOT = Path('/content/drive/MyDrive/YOLOv8_with_P2_workspace')
REPO_ON_DRIVE = DRIVE_ROOT / 'YOLOv8_with_P2'          # Thư mục code đã upload
DATASET_ON_DRIVE = DRIVE_ROOT / 'URPC2020.zip'         # Hoặc thư mục archive/URPC2020 trên Drive

PERSISTENT_RUNS = DRIVE_ROOT / 'runs/detect'
PERSISTENT_REPORTS = DRIVE_ROOT / 'reports'
PERSISTENT_RUNS.mkdir(parents=True, exist_ok=True)
PERSISTENT_REPORTS.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ON_DRIVE)
print('Working directory:', Path.cwd())
print('Persistent runs:', PERSISTENT_RUNS)


## Cài đặt môi trường GPU đã khóa phiên bản

Cài đặt `torch==2.9.1`, `torchvision==0.24.1` (CUDA build) và `requirements.txt`, sau đó chạy bộ unit test để xác minh môi trường.


In [ ]:
!nvidia-smi
%pip install -q torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cu126 || \
  pip install -q torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cu128
%pip install -q -r requirements.txt -r requirements-dev.txt
!python -m pytest -q


## Cổng D0: Chuẩn bị dữ liệu trên SSD `/content`, sửa hộp vượt biên và tách rò rỉ `train`/`val`

Bước này giữ nguyên tập `val` và `test`, loại bỏ ảnh trùng chính xác giữa `train` và `val`/`test` (ví dụ `000002.jpg`), xuất `configs/data_dev.yaml` và lưu báo cáo kiểm toán vào Google Drive.


In [ ]:
import shutil
import zipfile
import yaml

LOCAL_RAW_DIR = Path('/content/archive/URPC2020')
if not LOCAL_RAW_DIR.exists():
    LOCAL_RAW_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DATASET_ON_DRIVE.suffix == '.zip':
        with zipfile.ZipFile(DATASET_ON_DRIVE) as zf:
            zf.extractall('/content/archive')
    else:
        shutil.copytree(DATASET_ON_DRIVE, LOCAL_RAW_DIR)

# Tạo cấu hình raw cục bộ trỏ vào /content/archive/URPC2020
raw_colab_yaml = Path('configs/data_urpc_raw_colab.yaml')
if not raw_colab_yaml.exists():
    raw_colab_yaml.write_text(yaml.safe_dump({
        'path': str(LOCAL_RAW_DIR),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'names': {0: 'holothurian', 1: 'echinus', 2: 'scallop', 3: 'starfish'},
    }, sort_keys=False), encoding='utf-8')

# 1) Đảm bảo hộp vượt biên <=1px và hộp diện tích 0 đã được sửa an toàn
!python preprocessing_data/repair_labels.py --data configs/data_urpc_raw_colab.yaml --output "{PERSISTENT_REPORTS}/urpc-colab-repair" --apply

# 2) Tạo tập phát triển sạch rò rỉ chính xác (dùng hardlink để không tốn thêm dung lượng ổ đĩa)
dev_yaml = Path('configs/data_dev.yaml')
if not dev_yaml.exists():
    !python preprocessing_data/prepare_dev_split.py \
        --data configs/data_urpc_raw_colab.yaml \
        --output /content/datasets/URPC2020_dev \
        --yaml-output configs/data_dev.yaml \
        --report "{PERSISTENT_REPORTS}/data_decisions.md" \
        --link-mode hardlink

# 3) Chạy audit để khóa fingerprint và sinh groups_val.json
audit_dev_dir = PERSISTENT_REPORTS / 'audit-dev'
if not audit_dev_dir.exists():
    !python dataset_audit.py --data configs/data_dev.yaml --splits train val test --output "{audit_dev_dir}"


## Cổng D1: Chạy thử 3 Epochs trên GPU để đo VRAM, thời gian/epoch và kiểm tra `NaN`/`Inf`


In [ ]:
import json

!python train.py --baseline B1-Raw --data configs/data_dev.yaml --epochs 3 --name pilot-gpu-b1 --project "{PERSISTENT_RUNS}" --device 0

telemetry_file = PERSISTENT_RUNS / 'pilot-gpu-b1/training_telemetry.json'
print(json.dumps(json.loads(telemetry_file.read_text()), indent=2))


## Cổng D2: Huấn luyện Full Baselines (`B0-Raw` và `B1-Raw`, 100 epochs, seed 0)

- Nếu phiên Colab bị ngắt giữa chừng, chạy ô **Resume** ngay bên dưới để tiếp tục từ `weights/last.pt` mà không mất các epoch đã học.


In [ ]:
# Baseline 1: B0-Raw (YOLOv8n)
!python train.py --baseline B0-Raw --data configs/data_dev.yaml --seed 0 --name dev-B0-s0 --project "{PERSISTENT_RUNS}" --device 0
!python eval.py --data configs/data_dev.yaml --weights "{PERSISTENT_RUNS}/dev-B0-s0/weights/best.pt" --groups "{PERSISTENT_REPORTS}/audit-dev/groups_val.json" --output "{PERSISTENT_REPORTS}/eval-B0-s0" --device 0

# Baseline 2: B1-Raw (YOLOv8n-P2)
!python train.py --baseline B1-Raw --data configs/data_dev.yaml --seed 0 --name dev-B1-s0 --project "{PERSISTENT_RUNS}" --device 0
!python eval.py --data configs/data_dev.yaml --weights "{PERSISTENT_RUNS}/dev-B1-s0/weights/best.pt" --groups "{PERSISTENT_REPORTS}/audit-dev/groups_val.json" --output "{PERSISTENT_REPORTS}/eval-B1-s0" --device 0


In [ ]:
# (Chỉ chạy ô này khi Colab bị ngắt giữa chừng ở một lượt train đang dở)
# Đổi tên thư mục run cần khôi phục (ví dụ dev-B1-s0):
# !python train.py --resume "{PERSISTENT_RUNS}/dev-B1-s0/weights/last.pt" --device 0


## Đối chứng tiền xử lý: `B0-Enhanced` (Gray World + CLAHE)


In [ ]:
clahe_yaml = Path('configs/data_dev_clahe.yaml')
if not clahe_yaml.exists():
    !python preprocessing_data/clahe.py \
        --data configs/data_dev.yaml \
        --output /content/datasets/URPC2020_dev_clahe \
        --yaml-output configs/data_dev_clahe.yaml \
        --workers 4

!python train.py --baseline B0-Enhanced --data configs/data_dev_clahe.yaml --seed 0 --name dev-B0E-s0 --project "{PERSISTENT_RUNS}" --device 0
# Chấm B0-Enhanced trên chính các nhóm severity/clutter đã định nghĩa trên ảnh gốc (kiểm tra khớp hình học & nhãn):
!python eval.py --data configs/data_dev_clahe.yaml --weights "{PERSISTENT_RUNS}/dev-B0E-s0/weights/best.pt" --groups "{PERSISTENT_REPORTS}/audit-dev/groups_val.json" --allow-pixel-modified-groups --output "{PERSISTENT_REPORTS}/eval-B0E-s0" --device 0


## Tổng hợp bảng kết quả Cổng D2 (Overall, từng lớp, chi phí VRAM & thời gian)


In [ ]:
import json

summary_rows = []
for run_name, eval_name in [('dev-B0-s0', 'eval-B0-s0'), ('dev-B1-s0', 'eval-B1-s0'), ('dev-B0E-s0', 'eval-B0E-s0')]:
    metrics_path = PERSISTENT_REPORTS / eval_name / 'metrics.json'
    telemetry_path = PERSISTENT_RUNS / run_name / 'training_telemetry.json'
    if not metrics_path.is_file():
        continue
    metrics_data = json.loads(metrics_path.read_text())
    overall = next(r for r in metrics_data['metrics'] if r['group'] == 'all')
    telemetry = json.loads(telemetry_path.read_text()) if telemetry_path.is_file() else {}
    summary_rows.append({
        'run': run_name,
        'AP': round(overall['AP'], 4) if overall['AP'] is not None else None,
        'AP_small': round(overall['AP_small'], 4) if overall['AP_small'] is not None else None,
        'AR_small': round(overall['AR_small'], 4) if overall['AR_small'] is not None else None,
        'FP_per_img': round(overall['FP_per_image'], 3),
        'FN_small_rate': round(overall['FN_small_rate'], 4) if overall['FN_small_rate'] is not None else None,
        'hours': round(telemetry.get('total_training_seconds', 0) / 3600, 2),
        'peak_vram_mb': telemetry.get('max_peak_vram_allocated_mb'),
    })

for row in summary_rows:
    print(row)
